In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

from config import PROCESSED_DATA_PATH

In [37]:
def load_clean_data():
    path = Path(PROCESSED_DATA_PATH) / "clean_panel.parquet"
    df = pd.read_parquet(path)
    return df

df = load_clean_data()

In [38]:
df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume,return_1d,fwd_return_5d,fwd_return_21d
0,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09,0.001729,-0.031066,-0.104160
1,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09,-0.015906,-0.001517,-0.073518
2,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09,-0.001849,-0.005461,-0.078165
3,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09,0.006648,-0.028540,-0.074488
4,2010-01-11,AAPL,6.301419,7.503929,7.607143,7.444643,7.600000,462229600.0,3.468538e+09,-0.008821,0.023464,-0.071344
...,...,...,...,...,...,...,...,...,...,...,...,...
148559,2024-11-21,XOM,117.613434,121.930000,122.550003,120.269997,121.080002,14675400.0,1.789372e+09,0.013381,-0.032560,-0.128188
148560,2024-11-22,XOM,117.478386,121.790001,123.209999,121.639999,121.820000,13323400.0,1.622657e+09,-0.001148,-0.032351,-0.126365
148561,2024-11-25,XOM,115.722816,119.970001,121.879997,119.610001,121.430000,26580300.0,3.188839e+09,-0.014944,-0.019172,-0.112361
148562,2024-11-26,XOM,113.793625,117.970001,119.680000,117.849998,119.529999,14827300.0,1.749177e+09,-0.016671,-0.031279,-0.097398


------------------

FEATURE ENGINEERING

----------------

1) MOMENTUM FEATURES

In [28]:
# Now creating all the momentum features 

def momentum(df):
    
    df = df.copy()
    
    windows = [5, 10, 21, 63, 252]
    
    #Direct momentum features based on return
    for w in windows:
        
        df[f"return_{w}d"] = (df.groupby('ticker')["adj_close"].pct_change(w))
        
    # These features are a combination of both price and moving average, gives an idea how far price has gone from its mean
    for w in windows:
        
        ma = df.groupby('ticker')["adj_close"].transform(lambda x: x.rolling(w).mean())
        df[f"price_ma_ratio_{w}"] = (df["adj_close"] - ma)/ma
    
    #This is to get what percentage of days was my return greater than 0
    pos_days = (df.groupby('ticker')["return_1d"].transform(lambda x: x>0).rolling(21).mean())
    
    df['pos_days_21'] = pos_days
    
    return df
    

-------------

2) MEAN REVERSION FEATURES

---------------

In [29]:
def mean_reversion(df):
    
    df = df.copy()
    
    #Shift by 1 to prevent look-ahead bias(you want the comparison of current value to how values have changed before, so including today's is wrong)
    
    
    #Z-scores based on price
    roll_mean_21 = df.groupby('ticker')['adj_close'].transform(lambda x :x.shift(1).rolling(21).mean())
    
    roll_std_21 = df.groupby('ticker')['adj_close'].transform(lambda x :x.shift(1).rolling(21).std())
    
    df["z_price_21"] = (df['adj_close'] - roll_mean_21)/roll_std_21
    
    roll_mean_40 = df.groupby('ticker')['adj_close'].transform(lambda x :x.shift(1).rolling(40).mean())
    
    roll_std_40 = df.groupby('ticker')['adj_close'].transform(lambda x :x.shift(1).rolling(40).std())
    
    df["z_price_63"] = (df['adj_close'] - roll_mean_40)/roll_std_40
    
    #Z-scores based on 1d, 5d returns
    roll_mean_ret_5 = df.groupby('ticker')['return_1d'].transform(lambda x : x.shift(1).rolling(5).mean())
    roll_mean_ret_21 = df.groupby('ticker')['return_1d'].transform(lambda x : x.shift(1).rolling(21).mean())
    roll_mean_ret_35 = df.groupby('ticker')['return_1d'].transform(lambda x : x.shift(1).rolling(35).mean())
    
    roll_std_ret_5 = df.groupby('ticker')['return_1d'].transform(lambda x :x.shift(1).rolling(5).std())
    roll_std_ret_21 = df.groupby('ticker')['return_1d'].transform(lambda x :x.shift(1).rolling(21).std())
    roll_std_ret_35 = df.groupby('ticker')['return_1d'].transform(lambda x :x.shift(1).rolling(35).std())
    
    df["z_return_1d_5"] = (df['return_1d']-roll_mean_ret_5)/roll_std_ret_5
    
    df["z_return_1d_21"] = (df['return_1d']-roll_mean_ret_21)/roll_std_ret_21
    
    df["z_return_1d_35"] = (df['return_1d']-roll_mean_ret_35)/roll_std_ret_35
    
    roll_mean_ret_5_5d = df.groupby('ticker')['return_5d'].transform(lambda x : x.shift(1).rolling(5).mean())
    roll_mean_ret_21_5d = df.groupby('ticker')['return_5d'].transform(lambda x : x.shift(1).rolling(21).mean())
    roll_mean_ret_35_5d = df.groupby('ticker')['return_5d'].transform(lambda x : x.shift(1).rolling(35).mean())
    
    roll_std_ret_5_5d = df.groupby('ticker')['return_5d'].transform(lambda x :x.shift(1).rolling(5).std())
    roll_std_ret_21_5d = df.groupby('ticker')['return_5d'].transform(lambda x :x.shift(1).rolling(21).std())
    roll_std_ret_35_5d = df.groupby('ticker')['return_5d'].transform(lambda x :x.shift(1).rolling(35).std())
    
    df["z_return_5d_5"] = (df['return_5d']-roll_mean_ret_5_5d)/roll_std_ret_5_5d
    
    df["z_return_5d_21"] = (df['return_5d']-roll_mean_ret_21_5d)/roll_std_ret_21_5d
    
    df["z_return_5d_35"] = (df['return_5d']-roll_mean_ret_35_5d)/roll_std_ret_35_5d
    
    
    #Simple negative of return features
    df["rev_1d"] = -df["return_1d"]
    df["rev_5d"] = -df['return_5d']

    return df

In [ ]:
'''
def create_rolling_features(df, col, windows, stats=None, shift=1):

    if stats is None:
        stats = ["mean", "std"]
    
    stat_funcs = {
        "mean": lambda x, w: x.shift(shift).rolling(w).mean(),
        "std": lambda x, w: x.shift(shift).rolling(w).std(),
        "min": lambda x, w: x.shift(shift).rolling(w).min(),
        "max": lambda x, w: x.shift(shift).rolling(w).max(),
    }
    
    for window in windows:
        for stat in stats:
            col_name = f"{col}_{stat}_{window}d"
            df[col_name] = df.groupby("ticker")[col].transform(
                lambda x: stat_funcs[stat](x, window)
            )
    
    return df

# Usage - creates 6 features in one line!
df = create_rolling_features(
    df, 
    col="return_1d", 
    windows=[5, 21, 35],
    stats=["mean", "std"]
)
# Creates: return_1d_mean_5d, return_1d_std_5d, return_1d_mean_21d, etc.
'''

-------------


3) VOLATILITY AND RISK FEATURES

----------------

In [39]:
def volatility(df):
    
    df = df.copy()
    
    for w in [5,10,21,63]:
        
        df[f"vol_{w}"] = df.groupby('ticker')['return_1d'].transform(lambda x : x.rolling(w).std())
        
    df['vol_ratio_21_63'] =  df['vol_21']/ df['vol_63']
    
    df['vol_ratio_5_21'] = df['vol_5']/df['vol_21']
    
    df["hl_range"] = (df["high"] - df["low"]) / df["close"]

    downside_1d = df["return_1d"].clip(upper=0)
    
    downside_5d = df['return_5d'].clip(upper=0)
    
    df["downside_vol_1d"] = downside_1d.groupby(df["ticker"]).transform(
        lambda x: x.rolling(21).std()
    )

    df["downside_vol_5d"] = downside_5d.groupby(df["ticker"]).transform(
        lambda x: x.rolling(21).std()
    )
    
    return df
    

---------------

4) VOLUME AND LIQUIDITY FEATURES

-------------

In [40]:
def volume_features(df):
    
    df = df.copy()
    
    df['dollar_volume'] = df['adj_close']*df['volume']
    
    df['vol_avg_5'] = df.groupby('ticker')['volume'].transform(lambda x : x.rolling(5).mean())
    df['vol_avg_21'] = df.groupby('ticker')['volume'].transform(lambda x : x.rolling(21).mean())
    df['vol_avg_63'] = df.groupby('ticker')['volume'].transform(lambda x : x.rolling(63).mean())
    
    df["dollar_vol_log"] = np.log1p(df['dollar_volume'])
    
    #gives slope of the volume trend
    vol_slope = (   
        df.groupby("ticker")["volume"]
        .transform(lambda x: x.rolling(21).apply(
            lambda y: np.polyfit(np.arange(len(y)), y, 1)[0], raw=False))
    )

    df["volume_trend"] = vol_slope

    df["mom_x_vol_5"] = df["return_5d"] * df["vol_avg_5"]
    df["mom_x_vol_21"] = df["return_21d"] * df["vol_avg_21"]
    df["mom_x_vol_63"] = df["return_63d"] * df["vol_avg_63"]
    return df

Cluster	Name (for intuition)

1. 	Short-Term Momentum
2. 	Volume & Flow Momentum
3.	Short-Term Reversal
4.	Trend Positioning & Risk
5.	Volatility Term Structure
6.	Medium-Term Trend & Volatility

--------------

CROSS-SECTIONAL NORMALIZATION

--------------------

In [41]:
def cross_sectional_rank(df, feature_cols):
    
    df = df.copy()

    for col in feature_cols:
        df[f"{col}_rank"] = (
            df.groupby("date")[col]
            .rank(pct=True)
        )

    return df


In [53]:
def get_base_features(df):
    return [
        col for col in df.columns
        if col.startswith((
            "ret_", "price_ma", "pos_", "z_", "rev_",
            "vol_", "hl_", "downside", "vol_ratio",
            "dollar", "volume_", "mom_x_vol"
        ))
        and not col.endswith("_rank")
    ]


In [ ]:
def select_features(df):
    
    base_features = get_base_features(df)
    rank_features = [f"{c}_rank" for c in base_features]

    final_features = base_features + rank_features

    return df[["date", "ticker", "fwd_return_5d"] + final_features]

In [43]:
def save_features(df):
    
    path = Path(PROCESSED_DATA_PATH) / "features_df.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved features to {path}")

In [ ]:
df

,date,ticker,adj_close,close,high,low,open,volume,dollar_volume,return_1d,fwd_return_5d,fwd_return_21d
0,2010-01-05,AAPL,6.429479,7.656429,7.699643,7.616071,7.664286,601904800.0,4.608441e+09,0.001729,-0.031066,-0.104160
1,2010-01-06,AAPL,6.327210,7.534643,7.686786,7.526786,7.656429,552160000.0,4.160329e+09,-0.015906,-0.001517,-0.073518
2,2010-01-07,AAPL,6.315512,7.520714,7.571429,7.466071,7.562500,477131200.0,3.588367e+09,-0.001849,-0.005461,-0.078165
3,2010-01-08,AAPL,6.357501,7.570714,7.571429,7.466429,7.510714,447610800.0,3.388733e+09,0.006648,-0.028540,-0.074488
4,2010-01-11,AAPL,6.301419,7.503929,7.607143,7.444643,7.600000,462229600.0,3.468538e+09,-0.008821,0.023464,-0.071344
...,...,...,...,...,...,...,...,...,...,...,...,...
148559,2024-11-21,XOM,117.613434,121.930000,122.550003,120.269997,121.080002,14675400.0,1.789372e+09,0.013381,-0.032560,-0.128188
148560,2024-11-22,XOM,117.478386,121.790001,123.209999,121.639999,121.820000,13323400.0,1.622657e+09,-0.001148,-0.032351,-0.126365
148561,2024-11-25,XOM,115.722816,119.970001,121.879997,119.610001,121.430000,26580300.0,3.188839e+09,-0.014944,-0.019172,-0.112361
148562,2024-11-26,XOM,113.793625,117.970001,119.680000,117.849998,119.529999,14827300.0,1.749177e+09,-0.016671,-0.031279,-0.097398


In [55]:
def run_pipeline(df):
    
    df = load_clean_data()

    df = momentum(df)
    df = mean_reversion(df)
    df = volatility(df)
    df = volume_features(df)
    
    base_features = get_base_features(df)

    df = cross_sectional_rank(df, base_features)
    
    df = select_features(df)

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    save_features(df)
    
run_pipeline(df)

Saved features to /Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/features_df.parquet


In [58]:
def load_clean_data():
    
    path = Path(PROCESSED_DATA_PATH) / "features_df.parquet"
    df = pd.read_parquet(path)
    return df

df = load_clean_data()
df

,date,ticker,fwd_return_5d,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,...,downside_vol_1d_rank,downside_vol_5d_rank,vol_avg_5_rank,vol_avg_21_rank,vol_avg_63_rank,dollar_vol_log_rank,volume_trend_rank,mom_x_vol_5_rank,mom_x_vol_21_rank,mom_x_vol_63_rank
0,2011-01-03,AAPL,0.039081,4.399815e+09,0.013095,0.015186,0.022646,0.053737,0.266111,0.571429,...,0.078947,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.973684
1,2011-01-04,AAPL,0.031242,3.070944e+09,0.014752,0.017638,0.025883,0.056965,0.270454,0.619048,...,0.052632,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.947368
2,2011-01-05,AAPL,0.031198,2.559542e+09,0.017622,0.022883,0.032167,0.063198,0.278453,0.619048,...,0.078947,0.078947,0.947368,0.947368,0.973684,0.973684,0.078947,0.947368,0.947368,0.947368
3,2011-01-06,AAPL,0.035807,3.006965e+09,0.010599,0.019381,0.028983,0.059955,0.275035,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,1.000000,0.052632,0.947368,0.947368,0.947368
4,2011-01-07,AAPL,0.036773,3.144450e+09,0.009545,0.022770,0.034058,0.065287,0.281753,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,0.973684,0.052632,0.947368,0.947368,0.947368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138519,2024-11-21,XOM,-0.032560,1.726024e+09,0.015237,0.015643,0.026965,0.038791,0.105601,0.476190,...,0.150000,0.325000,0.575000,0.625000,0.625000,0.625000,0.375000,0.725000,0.625000,0.700000
138520,2024-11-22,XOM,-0.032351,1.565212e+09,0.009901,0.013070,0.024482,0.037044,0.103489,0.476190,...,0.150000,0.350000,0.600000,0.625000,0.625000,0.525000,0.275000,0.600000,0.625000,0.550000
138521,2024-11-25,XOM,-0.019172,3.075947e+09,-0.004630,-0.002469,0.008585,0.021099,0.086262,0.476190,...,0.250000,0.375000,0.600000,0.625000,0.625000,0.675000,0.475000,0.150000,0.450000,0.525000
138522,2024-11-26,XOM,-0.031279,1.687252e+09,-0.020150,-0.017956,-0.008245,0.003751,0.067485,0.476190,...,0.275000,0.375000,0.600000,0.625000,0.625000,0.750000,0.450000,0.175000,0.375000,0.425000


In [62]:
df[40:80]

,date,ticker,fwd_return_5d,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,...,downside_vol_1d_rank,downside_vol_5d_rank,vol_avg_5_rank,vol_avg_21_rank,vol_avg_63_rank,dollar_vol_log_rank,volume_trend_rank,mom_x_vol_5_rank,mom_x_vol_21_rank,mom_x_vol_63_rank
40,2011-03-02,AAPL,0.000994,6.363629e+09,0.008547,0.006374,0.002796,0.045695,0.245027,0.666667,...,0.657895,0.842105,0.973684,0.973684,0.973684,1.000000,0.973684,1.000000,1.000000,0.973684
41,2011-03-03,AAPL,-0.035850,5.400292e+09,0.020109,0.028687,0.021971,0.065621,0.268684,0.666667,...,0.657895,0.842105,0.973684,0.973684,0.973684,1.000000,0.973684,1.000000,1.000000,0.973684
42,2011-03-04,AAPL,-0.022250,4.893818e+09,0.014541,0.029445,0.021054,0.064828,0.267732,0.714286,...,0.657895,0.815789,0.973684,0.973684,0.973684,1.000000,0.947368,1.000000,1.000000,0.973684
43,2011-03-07,AAPL,-0.005065,5.820373e+09,0.000254,0.014784,0.006274,0.049237,0.249015,0.714286,...,0.657895,0.815789,0.973684,0.973684,0.973684,1.000000,0.973684,1.000000,1.000000,0.973684
44,2011-03-08,AAPL,-0.029036,3.801763e+09,-0.002244,0.010975,0.006151,0.048667,0.248110,0.714286,...,0.657895,0.815789,0.973684,0.973684,0.973684,1.000000,0.052632,1.000000,1.000000,0.973684
45,2011-03-09,AAPL,-0.063722,4.791873e+09,-0.011665,-0.001170,-0.003233,0.037307,0.234374,0.666667,...,0.631579,0.815789,0.947368,0.973684,0.973684,1.000000,0.052632,0.789474,0.842105,0.973684
46,2011-03-10,AAPL,-0.034701,5.276882e+09,-0.020850,-0.018661,-0.018508,0.019016,0.212021,0.619048,...,0.631579,0.789474,0.947368,0.973684,0.973684,1.000000,0.052632,0.052632,0.105263,0.973684
47,2011-03-11,AAPL,-0.060570,4.972982e+09,-0.001305,-0.004680,-0.002616,0.033100,0.228484,0.619048,...,0.631579,0.763158,0.947368,0.973684,0.973684,1.000000,0.052632,0.078947,0.131579,0.973684
48,2011-03-14,AAPL,-0.040333,4.622731e+09,0.004175,-0.000339,0.001965,0.036115,0.231751,0.666667,...,0.657895,0.763158,0.947368,0.973684,0.973684,1.000000,0.131579,0.105263,0.263158,0.973684
49,2011-03-15,AAPL,-0.012246,7.470261e+09,-0.013125,-0.022254,-0.019564,0.011172,0.201417,0.619048,...,0.657895,0.736842,0.947368,0.973684,0.973684,1.000000,0.789474,0.052632,0.078947,0.947368


------
